<a href="https://colab.research.google.com/github/kirti2k23/Finance-AI-Assistant/blob/main/notebooks/non_instruction_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1: Non-Instruction Fine-Tuning
**Goal:** Adapt base model to finance domain language before instruction tuning.

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## Step 2 — Imports

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import re

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4
VRAM: 15.6 GB


## Step 3 — Configuration

In [ ]:
# ── Model config ──
MODEL_NAME    = "unsloth/Llama-3.2-1B"
MAX_SEQ_LEN   = 512
LOAD_IN_4BIT  = True   # QLoRA — keeps VRAM under 8GB

# ── LoRA config ──
LORA_R        = 16
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
TARGET_MODS   = ["q_proj", "k_proj", "v_proj", "o_proj",
                 "up_proj", "down_proj", "gate_proj"]

# ── Training config ──
OUTPUT_DIR    = "./outputs/stage1_non_instruction"
EPOCHS        = 3
BATCH_SIZE    = 2
GRAD_ACCUM    = 4
LR            = 2e-4

## Step 4 — Load base model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,          # Auto-detect (bfloat16 on A100, float16 on T4)
    load_in_4bit    = LOAD_IN_4BIT,
)
print("Base model loaded successfully")

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.


Base model loaded successfully


## Step 5 — Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                    = LORA_R,
    target_modules       = TARGET_MODS,
    lora_alpha           = LORA_ALPHA,
    lora_dropout         = LORA_DROPOUT,
    bias                 = "none",
    use_gradient_checkpointing = "unsloth",
    random_state         = 42,
)
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.2 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


## Step 6 — Load and prepare raw finance text

In [ ]:
from datasets import load_dataset

# Load finance dataset from HuggingFace
ds = load_dataset("gbharti/finance-alpaca")
print(ds)
print("Sample record:", ds["train"][0])

# Extract output field as raw finance text paragraphs
# 'output' contains domain knowledge — perfect for non-instruction FT
paragraphs = [
    item["output"].strip()
    for item in ds["train"]
    if len(item["output"].strip()) > 100  # filter out very short responses
]

# Take first 500 paragraphs — more than enough for non-instruction FT
paragraphs = paragraphs[:500]
print(f"\nTotal paragraphs extracted: {len(paragraphs)}")
print("\nSample paragraph:")
print(paragraphs[0])

README.md:   0%|          | 0.00/831 [00:00<?, ?B/s]

Cleaned_date.json:   0%|          | 0.00/42.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/68912 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 68912
    })
})
Sample record: {'instruction': 'For a car, what scams can be plotted with 0% financing vs rebate?', 'input': '', 'output': "The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options

## Step 7 — Tokenize and chunk text

In [ ]:
EOS = tokenizer.eos_token

def chunk_text(paragraphs, tokenizer, max_len=MAX_SEQ_LEN):
    chunks = []
    current_chunk = ""
    for para in paragraphs:
        # NEW: if single paragraph exceeds max_len, truncate it
        para_toks = tokenizer(para, return_tensors="pt")["input_ids"].shape[1]
        if para_toks > max_len:
            tokens = tokenizer(para, return_tensors="pt")["input_ids"][0][:max_len-10]
            para = tokenizer.decode(tokens, skip_special_tokens=True)

        candidate = current_chunk + " " + para if current_chunk else para
        toks = tokenizer(candidate, return_tensors="pt")["input_ids"].shape[1]
        if toks <= max_len:
            current_chunk = candidate
        else:
            if current_chunk:
                chunks.append(current_chunk + EOS)
            current_chunk = para

    if current_chunk:
        chunks.append(current_chunk + EOS)
    return chunks

chunks = chunk_text(paragraphs, tokenizer, max_len=506)
print(f"Total training chunks: {len(chunks)}")

dataset = Dataset.from_dict({"text": chunks})
print(dataset)

Total training chunks: 265
Dataset({
    features: ['text'],
    num_rows: 265
})


In [ ]:
dataset['text']

Column(["The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0% interest loan. These tend to be shorter 12 months vs 36,48,60 or even 72 months. The shorter length makes it harder for many to afford. If you can't swing the 12 large payments they offer yo

In [ ]:
# Extract real prompts from your actual training chunks
test_prompts = [
    " ".join(chunks[0].split()[:6]),
    " ".join(chunks[80].split()[:6]),
    " ".join(chunks[200].split()[:6]),
]

print("Prompts taken from training data:")
for i, p in enumerate(test_prompts):
    print(f"  {i+1}. {p}")

Prompts taken from training data:
  1. The car deal makes money 3
  2. It certainly is possible for a
  3. New clothes isn't exactly an emergency


In [ ]:
# TEST BEFORE TRAINING = base model behavior
FastLanguageModel.for_inference(model)

print("=" * 60)
print("BASE MODEL OUTPUTS (before training)")
print("=" * 60)

base_results = {}
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 80,
        temperature    = 0.7,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    continuation = generated[len(prompt):].strip()
    base_results[prompt] = continuation
    print(f"\nPrompt    : {prompt}")
    print(f"Continues : {continuation}")
    print("-" * 60)

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL OUTPUTS (before training)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


Prompt    : The car deal makes money 3
Continues : ,000 miles away from the car, in the form of a loan that the car dealer will pay back to the car buyer. In this case, the car dealer will loan the car buyer $5,000, and the car buyer will pay back the loan over a 60-month period.
How much money does the car dealer make?
The car dealer makes money by charging a fee to the car
------------------------------------------------------------


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt    : It certainly is possible for a
Continues : person to get a job in the fashion industry. It is not that hard. It is only that most people do not know how to get the job. In fact, if you do not know how to get a job, it is possible that you will not get a job. That is why it is important to know how to get a job in the fashion industry. This article will help you to
------------------------------------------------------------

Prompt    : New clothes isn't exactly an emergency
Continues : , but we can't wait that long to get our favorite new outfits. We can't wait to put on our new outfits, but we can't wait to put on our new outfits for a while. This is what we're talking about when we talk about the new clothes we're wearing.
We are talking about the new clothes we are wearing. We are talking about the new clothes we are wearing
------------------------------------------------------------


## Step 8 — Train

In [ ]:
trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LEN,
    args = TrainingArguments(
        output_dir              = OUTPUT_DIR,
        num_train_epochs        = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate           = LR,
        fp16                    = not torch.cuda.is_bf16_supported(),
        bf16                    = torch.cuda.is_bf16_supported(),
        logging_steps           = 10,
        save_strategy           = "no",       # Save after every epoch
        warmup_steps            = 10,
        lr_scheduler_type       = "cosine",
        report_to               = "none",
    ),
)

print("Starting non-instruction fine-tuning...")
trainer_stats = trainer.train()
print(f"Training complete. Final loss: {trainer_stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/265 [00:00<?, ? examples/s]

Starting non-instruction fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 265 | Num Epochs = 3 | Total steps = 102
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
10,2.649994
20,2.596805
30,2.704524
40,2.649845
50,2.443825
60,2.476288
70,2.462110
80,2.307882
90,2.351535
100,2.253455


Training complete. Final loss: 2.4866


## Step 9 — Save adapter

In [ ]:
ADAPTER_PATH = "./outputs/stage1_adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Stage 1 adapter saved to: {ADAPTER_PATH}")

# Also save to Google Drive if available
try:
    from google.colab import drive
    drive.mount("/content/drive")
    model.save_pretrained("/content/drive/MyDrive/finance_ft/stage1_adapter")
    print("Adapter also saved to Google Drive")
except:
    print("Not on Colab — skipping Drive save")

Unsloth: Restored added_tokens_decoder metadata in ./outputs/stage1_adapter/tokenizer_config.json.


Stage 1 adapter saved to: ./outputs/stage1_adapter
Mounted at /content/drive
Adapter also saved to Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from unsloth import FastLanguageModel

ADAPTER_PATH = "/content/drive/MyDrive/finance_ft/stage1_adapter"
MAX_SEQ_LEN  = 512
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = ADAPTER_PATH,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)

print("Stage 1 adapter loaded successfully from Drive")

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.


Stage 1 adapter loaded successfully from Drive


##Step 10 — Test model after non-instruction fine-tuning vs base model


In [ ]:
FastLanguageModel.for_inference(model)

print("=" * 60)
print("FINE-TUNED MODEL OUTPUTS (after Stage 1 training)")
print("=" * 60)

ft_results = {}
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 80,
        temperature    = 0.7,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    continuation = generated[len(prompt):].strip()
    ft_results[prompt] = continuation
    print(f"\nPrompt    : {prompt}")
    print(f"Continues : {continuation}")
    print("-" * 60)

# Side by side comparison
print("\n" + "=" * 60)
print("COMPARISON")
print("=" * 60)
for prompt in test_prompts:
    print(f"\nPROMPT     : {prompt}")
    print(f"BASE       : {base_results[prompt][:200]}")
    print(f"FINE-TUNED : {ft_results[prompt][:200]}")
    print("=" * 60)

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINE-TUNED MODEL OUTPUTS (after Stage 1 training)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt    : The car deal makes money 3
Continues : ways. First, they will buy your car for less than market value, assuming you owe more than market value. Second, they will finance the price you pay for the car, which they will do with interest. Third, they will offer you a trade-in, which they will offer you if they know you are financing. If you finance, the trade-in is a negative factor, and vice versa
------------------------------------------------------------


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt    : It certainly is possible for a
Continues : broker to make a profit by holding your order.  You are not guaranteed that the broker will hold your order.  If you are not sure that the broker will hold your order, then you should not be using a broker.  You should be using a market maker.  A market maker is a broker that is willing to take your order at any price.  In other words, if you
------------------------------------------------------------

Prompt    : New clothes isn't exactly an emergency
Continues : expense.  You can get a new set of clothes every month, and if you can't, you have bigger problems.  If you need a new car, you're in trouble.  You're more likely to be able to get a new car by making a down payment and paying it off over time than you are by buying a new car and financing it.  If you can't do
------------------------------------------------------------

COMPARISON

PROMPT     : The car deal makes money 3
BASE       : ,000 miles away from the car, in th